# Train STARK-ST

Stage 1 trains the backbone, transformer and box head; stage 2 the classification head. Original STARK settings,
any combination of datasets, one GPU. Details: [docs/train.md](../docs/train.md)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # Colab only: get / update the code
    os.system("git clone -q https://github.com/balk21/stark-cls-finetune /content/stark-cls-finetune || git -C /content/stark-cls-finetune pull -q")
    os.chdir("/content/stark-cls-finetune/notebooks")
sys.path.insert(0, os.getcwd())
import importlib, nbhelper; nb = importlib.reload(nbhelper)
nb.init(colab_drive="/content/drive/MyDrive/LOKAP")  # finds / creates the vot1 environment; colab_drive: Drive folder (Colab only)

## Parameters

In [ ]:
TRAIN = dict(
    name=None,                     # run folder <train_outputs>/<name>; None = generated from the parameters
    model_config="baseline_R101",  # "baseline_R101" (STARK-ST101) | "baseline" (STARK-ST50)
    stage=1,                       # 1: backbone + transformer + box head (500 epochs) | 2: classification head (50 epochs)
    init=None,                     # stage 1: None | stage 2 (required): "<stage-1 run name>" | "official" (STARK's weights) | path
    datasets=["got10k"],           # any combination of "got10k", "coco", "lasot", "trackingnet" ("got10k_full": all GOT-10k videos)
    dataset_ratios=None,           # sampling weight per dataset, e.g. [1, 2]; None = equal
    val_datasets=["got10k"],       # ["got10k"]: GOT-10k votval split | []: no validation
    epochs=None,                   # None = original: 500 (stage 1) / 50 (stage 2)
    lr_drop_epoch=None,            # learning rate / 10 after this epoch; None = 400 / 40
    samples_per_epoch=None,        # None = 60000
    val_samples_per_epoch=None,    # None = 10000
    val_interval=None,             # validate every N epochs; None = 20 / 10
    effective_batch=128,           # samples per optimizer step; 128 = original (8 GPUs x 16)
    micro_batch=16,                # samples per GPU pass (gradients are accumulated); lower it on "CUDA out of memory"
    num_workers=8,                 # data loading processes; lower it if RAM runs out
    seed=42,                       # STARK default
    keep_every=None,               # keep the weights every N epochs; None = 50 / 10
)

## Data

In [ ]:
nb.download_checkpoints([TRAIN["model_config"]])   # official checkpoint; only needed for init="official"
nb.prepare_training_data(TRAIN, got10k_sources=[])  # COCO: downloaded | GOT-10k: links or archive paths, [] = archive folder (docs/train.md)
nb.describe_training(TRAIN)                         # checks parameters and data; shows run folder, epochs, steps

## Train

In [ ]:
RUN = nb.train(TRAIN)  # same parameters again: resumes from the last finished epoch | overwrite=True: starts again

## Progress

In [ ]:
nb.show_training(TRAIN)  # losses and history plot; also works for a run that is still going (from another session)
nb.list_training_runs()  # all runs in <train_outputs>